# Native CLM Architecture Lab — cross-seed replication

Architecture search is frozen. This notebook retrains **T1 / M4 / X3 / X4** on development seeds **91002 / 91003**. Confirmation seeds **91101–91103 remain untouched**. Per-seed and cross-seed SVG evidence are generated automatically.


In [ ]:
BRANCH = "research/native-clm-lab"
PROFILE = "baseline"
REPLICATION_SEEDS = [91002, 91003]
RUN_REPLICATION = True
ALLOW_CPU = False
PUSH_DEV_RESULT = True
assert REPLICATION_SEEDS == [91002, 91003]


In [ ]:
from pathlib import Path
import base64, json, os, shutil, subprocess, sys

def run_checked(cmd, **kwargs):
    return subprocess.run(cmd, check=True, text=True, **kwargs)

repo_candidates = [Path.cwd(), Path("/kaggle/working/mini-cells")]
REPO_ROOT = next((p for p in repo_candidates if (p / ".git").exists()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path("/kaggle/working/mini-cells")
    run_checked(["git","clone","--depth","1","--branch",BRANCH,"https://github.com/ArcheLabs/mini-cells.git",str(REPO_ROOT)])
elif Path("/kaggle/working").exists():
    run_checked(["git","fetch","origin",BRANCH], cwd=REPO_ROOT)
    run_checked(["git","switch",BRANCH], cwd=REPO_ROOT)
    run_checked(["git","pull","--ff-only","origin",BRANCH], cwd=REPO_ROOT)
run_checked([sys.executable,"-m","pip","install","-q","-e",str(REPO_ROOT)+"[lm]"])
WORK_ROOT = Path("/kaggle/working/native-clm") if Path("/kaggle/working").exists() else REPO_ROOT/".native-clm-work"
CACHE_ROOT, OUTPUT_ROOT = WORK_ROOT/"cache", WORK_ROOT/"runs"
CACHE_ROOT.mkdir(parents=True, exist_ok=True); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("HEAD:", subprocess.check_output(["git","rev-parse","--short","HEAD"], cwd=REPO_ROOT, text=True).strip())


In [ ]:
def read_secret(name):
    if os.environ.get(name): return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
HF_TOKEN = read_secret("HF_TOKEN"); GITHUB_TOKEN = read_secret("GITHUB_TOKEN")
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print({"HF_TOKEN_available": bool(HF_TOKEN), "GITHUB_TOKEN_available": bool(GITHUB_TOKEN)})


## Frozen protocol gates

The test suite asserts exact seeds/models, parameter matching, finite forward passes, and dependency-free SVG generation before any 10M-token training starts.


In [ ]:
sys.path.insert(0, str(REPO_ROOT/"research/native-clm"))
from native_clm_replication import REPLICATION_MODELS, REPLICATION_SEEDS as FROZEN_SEEDS, parameter_summary
assert tuple(REPLICATION_SEEDS) == tuple(FROZEN_SEEDS)
run_checked([sys.executable,"-m","pytest","-q",str(REPO_ROOT/"research/native-clm/test_native_clm_replication.py")], cwd=REPO_ROOT)
print(json.dumps(parameter_summary(), indent=2))


## Run replication

With two T4 GPUs each seed runs two rounds: **T1/M4**, then **X3/X4**. T1 and M4 are retrained for every seed; seed-91001 anchors are not reused.


In [ ]:
RUNNER = REPO_ROOT/"research/native-clm/run_native_clm_replication.py"
cmd = [sys.executable, str(RUNNER), "sweep", "--profile", PROFILE, "--seeds", *[str(s) for s in REPLICATION_SEEDS], "--cache-root", str(CACHE_ROOT), "--output-root", str(OUTPUT_ROOT)]
if ALLOW_CPU: cmd.append("--allow-cpu")
if RUN_REPLICATION: run_checked(cmd, cwd=REPO_ROOT)
else: print("RUN_REPLICATION=False; existing outputs only")


## Cross-seed tables and visualizations


In [ ]:
seed_label = "-".join(str(s) for s in REPLICATION_SEEDS)
RUN_DIR = OUTPUT_ROOT/f"replication-{PROFILE}-seeds-{seed_label}"
summary = json.loads((RUN_DIR/"replication-summary.json").read_text(encoding="utf-8"))
try:
    import pandas as pd
    display(pd.DataFrame(summary["aggregate"]).sort_values("validation_ppl_mean"))
except Exception:
    print(json.dumps(summary["aggregate"], indent=2))
from IPython.display import SVG, display
for seed in REPLICATION_SEEDS:
    seed_dir = RUN_DIR/f"seed-{seed}"
    for filename in (f"replication-seed-{seed}-final-ppl.svg", f"replication-seed-{seed}-learning-curves.svg", f"replication-seed-{seed}-quality-compute.svg"):
        print(filename); display(SVG(filename=str(seed_dir/filename)))
for filename in ("replication-cross-seed-ppl.svg", "replication-paired-delta-vs-t1.svg"):
    print(filename); display(SVG(filename=str(RUN_DIR/filename)))


## Record and push replication evidence

Only JSON/CSV/SVG summaries are committed. Raw checkpoints and optimizer state remain outside Git.


In [ ]:
record_dir = REPO_ROOT/"research/native-clm/results/dev"/f"replication-{PROFILE}-seeds-{seed_label}"
if record_dir.exists(): shutil.rmtree(record_dir)
record_dir.mkdir(parents=True, exist_ok=True)
for src in RUN_DIR.rglob("*"):
    if src.is_file() and src.suffix in {".json",".csv",".svg"}:
        dst = record_dir/src.relative_to(RUN_DIR); dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
(record_dir/"README.md").write_text("# Native CLM cross-seed development replication\n\nFrozen models: T1/M4/X3/X4. Seeds: 91002/91003. Confirmation seeds 91101-91103 untouched.\n", encoding="utf-8")
print("evidence:", record_dir)


In [ ]:
def commit_and_push(repo, record_dir, branch, token):
    rel = record_dir.relative_to(repo)
    run_checked(["git","config","user.name","native-clm-kaggle"], cwd=repo)
    run_checked(["git","config","user.email","native-clm@users.noreply.github.com"], cwd=repo)
    run_checked(["git","add",str(rel)], cwd=repo)
    if subprocess.run(["git","diff","--cached","--quiet"], cwd=repo).returncode != 0:
        run_checked(["git","commit","-m","research: record Native CLM cross-seed replication"], cwd=repo)
    head = subprocess.check_output(["git","rev-parse","HEAD"], cwd=repo, text=True).strip()
    if not token:
        print("GITHUB_TOKEN unavailable; evidence committed locally only:", head); return
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy(); env["GIT_TERMINAL_PROMPT"] = "0"; env["GIT_CONFIG_COUNT"] = "1"; env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"; env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {basic}"
    run_checked(["git","push","https://github.com/ArcheLabs/mini-cells.git",f"HEAD:{branch}"], cwd=repo, env=env)
    print("pushed replication evidence:", head)
if PUSH_DEV_RESULT: commit_and_push(REPO_ROOT, record_dir, BRANCH, GITHUB_TOKEN)
